# Beginner SciPy Statistics Lab

This lab starts from the beginning: load a small dataset, clean it, inspect it, and then use SciPy for basic statistics. The goal is to understand what each statistic means before moving into the larger Module 5 notebooks.

## Important: Use a Python Notebook Runtime

Run this notebook with a **Python 3** kernel in Jupyter, VS Code, JupyterLab, or Google Colab. Do not run these cells in SQL Server, DBeaver, Azure Data Studio SQL query mode, or `sqlcmd`.

If you see SQL Server messages such as `Incorrect syntax near '%'`, the notebook code is being executed by a SQL engine instead of Python.

These beginner notebooks (`05`, `06`, and `07`) do **not** require a database connection. They load `data/beginner_financial_indicators.csv` when the file is present, and automatically create a small fallback dataset when the file is not present, which makes them work in Google Colab.


## 0. Setup

Run this cell first. It installs the required packages when the notebook is opened in a fresh environment such as Google Colab.

In [1]:
%pip install pandas numpy scipy matplotlib seaborn scikit-learn joblib -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
from scipy import stats

from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

In [3]:
DATA_PATH = Path("data/beginner_financial_indicators.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
def make_fallback_dataset():
    """Create a small dataset only when the CSV file is not available, such as in Colab."""
    dates = pd.date_range("2026-01-01", periods=10, freq="MS")
    templates = [
        ("Bank A", "Maseru", "Commercial Bank", 36.0, 19.5, 3.1, 5.0, 980, 610, 3100),
        ("Bank B", "Leribe", "Commercial Bank", 31.5, 17.4, 4.8, 4.2, 860, 590, 2500),
        ("Bank C", "Mafeteng", "Microfinance", 27.0, 14.8, 6.5, 3.1, 520, 380, 1700),
        ("Bank D", "Quthing", "Microfinance", 23.0, 12.4, 8.2, 2.2, 430, 350, 1350),
    ]
    rows = []
    for month_index, date in enumerate(dates):
        for inst_index, item in enumerate(templates):
            name, region, inst_type, liq, cap, npl, profit, dep, loans, tx = item
            liquidity = liq - month_index * (0.35 + inst_index * 0.08)
            capital = cap - month_index * (0.10 + inst_index * 0.02)
            npl_ratio = npl + month_index * (0.18 + inst_index * 0.05)
            stress = int((liquidity < 24) or (capital < 12) or (npl_ratio > 8.8))
            rows.append({
                "date": date,
                "institution": name,
                "region": region,
                "institution_type": inst_type,
                "liquidity_ratio": round(liquidity, 2),
                "capital_adequacy_ratio": round(capital, 2),
                "npl_ratio": round(npl_ratio, 2),
                "profit_margin": round(profit - npl_ratio * 0.12, 2),
                "total_deposits_m": dep + month_index * (8 - inst_index * 2),
                "total_loans_m": loans + month_index * (6 + inst_index * 2),
                "transaction_count": tx + month_index * (55 - inst_index * 5),
                "stress_flag": stress,
            })
    return pd.DataFrame(rows)


if DATA_PATH.exists():
    # parse_dates tells pandas that the date column should behave like a date, not plain text.
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
else:
    df = make_fallback_dataset()

print("Rows loaded:", len(df))
display(df.head())

Rows loaded: 41


,date,institution,region,institution_type,liquidity_ratio,capital_adequacy_ratio,npl_ratio,profit_margin,total_deposits_m,total_loans_m,transaction_count,stress_flag
0,2026-01-01,Bank A,Maseru,Commercial Bank,35.60,19.5,3.20,4.62,980,610,3100,0
1,2026-01-01,Bank B,Leribe,Commercial Bank,31.10,17.4,4.90,3.61,860,590,2500,0
2,2026-01-01,Bank C,Mafeteng,Microfinance,26.60,14.8,6.60,2.31,520,380,1700,0
3,2026-01-01,Bank D,Quthing,Microfinance,22.60,12.4,8.30,1.20,430,350,1350,1
4,2026-02-01,Bank A,Maseru,Commercial Bank,35.65,19.4,3.28,4.58,988,616,3155,0


## 1. Basic Data Cleaning

Real data often has duplicate rows, missing values, and columns that need the correct data type. Clean first so the statistics are based on reliable values.

In [5]:
print("Rows before cleaning:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("Missing values before cleaning:")
display(df.isna().sum())

# Work on a copy so the raw data stays unchanged.
clean_df = df.drop_duplicates().copy()

numeric_cols = [
    "liquidity_ratio",
    "capital_adequacy_ratio",
    "npl_ratio",
    "profit_margin",
    "total_deposits_m",
    "total_loans_m",
    "transaction_count",
]

# Convert numeric columns safely. Bad values become NaN so they can be filled.
for column in numeric_cols:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

# Fill missing numeric values with the median. The median is simple and less affected by outliers.
clean_df[numeric_cols] = clean_df[numeric_cols].fillna(clean_df[numeric_cols].median())

print("Rows after cleaning:", len(clean_df))
print("Missing values after cleaning:")
display(clean_df.isna().sum())
display(clean_df.head())

Rows before cleaning: 41
Duplicate rows: 1
Missing values before cleaning:


date                      0
institution               0
region                    0
institution_type          0
liquidity_ratio           1
capital_adequacy_ratio    0
npl_ratio                 1
profit_margin             1
total_deposits_m          0
total_loans_m             0
transaction_count         0
stress_flag               0
dtype: int64

Rows after cleaning: 40
Missing values after cleaning:


date                      0
institution               0
region                    0
institution_type          0
liquidity_ratio           0
capital_adequacy_ratio    0
npl_ratio                 0
profit_margin             0
total_deposits_m          0
total_loans_m             0
transaction_count         0
stress_flag               0
dtype: int64

,date,institution,region,institution_type,liquidity_ratio,capital_adequacy_ratio,npl_ratio,profit_margin,total_deposits_m,total_loans_m,transaction_count,stress_flag
0,2026-01-01,Bank A,Maseru,Commercial Bank,35.60,19.5,3.20,4.62,980,610,3100,0
1,2026-01-01,Bank B,Leribe,Commercial Bank,31.10,17.4,4.90,3.61,860,590,2500,0
2,2026-01-01,Bank C,Mafeteng,Microfinance,26.60,14.8,6.60,2.31,520,380,1700,0
3,2026-01-01,Bank D,Quthing,Microfinance,22.60,12.4,8.30,1.20,430,350,1350,1
4,2026-02-01,Bank A,Maseru,Commercial Bank,35.65,19.4,3.28,4.58,988,616,3155,0


## 2. First EDA Summary

EDA means exploratory data analysis. Start with simple row counts, averages, minimums, and maximums before using more advanced tests.

In [6]:
# describe().T gives one row per numeric column, which is easier to read.
display(clean_df[numeric_cols].describe().T.round(2))

# Group means answer: how do institutions differ on average?
institution_summary = clean_df.groupby("institution")[numeric_cols].mean().round(2)
display(institution_summary)

,count,mean,std,min,25%,50%,75%,max
liquidity_ratio,40.0,27.11,5.48,17.29,22.55,26.60,31.44,35.70
capital_adequacy_ratio,40.0,15.44,2.84,10.96,13.26,15.56,17.70,19.50
npl_ratio,40.0,6.79,2.31,3.20,4.88,6.60,8.56,11.27
profit_margin,40.0,2.73,1.32,0.58,1.80,3.09,3.74,4.62
total_deposits_m,40.0,720.00,242.10,430.00,502.00,708.00,930.50,1052.00
total_loans_m,40.0,523.00,113.46,350.00,417.50,530.00,631.00,664.00
transaction_count,40.0,2376.25,730.48,1350.00,1707.50,2302.50,2987.50,3595.00


,liquidity_ratio,capital_adequacy_ratio,npl_ratio,profit_margin,total_deposits_m,total_loans_m,transaction_count
institution,,,,,,,
Bank A,34.39,19.05,3.92,4.40,1016.0,637.0,3347.5
Bank B,29.08,16.86,5.85,3.36,887.0,626.0,2725.0
Bank C,24.66,14.17,7.69,2.03,538.0,425.0,1902.5
Bank D,20.30,11.68,9.70,1.13,439.0,404.0,1530.0


## 3. SciPy Descriptive Statistics

`stats.describe()` summarizes one numeric variable. Here we use liquidity ratio because it is a common financial health indicator.

In [7]:
liquidity = clean_df["liquidity_ratio"].to_numpy()
liquidity_stats = stats.describe(liquidity)

print("Number of observations:", liquidity_stats.nobs)
print("Minimum and maximum:", liquidity_stats.minmax)
print("Mean:", round(liquidity_stats.mean, 2))
print("Variance:", round(liquidity_stats.variance, 2))
print("Skewness:", round(liquidity_stats.skewness, 2))
print("Kurtosis:", round(liquidity_stats.kurtosis, 2))

Number of observations: 40
Minimum and maximum: (np.float64(17.29), np.float64(35.7))
Mean: 27.11
Variance: 30.07
Skewness: 0.03
Kurtosis: -1.16


## 4. Confidence Interval

A confidence interval gives a reasonable range for the true average. This example calculates a 95% confidence interval for liquidity ratio.

In [8]:
mean_liquidity = np.mean(liquidity)
standard_error = stats.sem(liquidity)

confidence_interval = stats.t.interval(
    confidence=0.95,
    df=len(liquidity) - 1,
    loc=mean_liquidity,
    scale=standard_error,
)

print("Mean liquidity:", round(mean_liquidity, 2))
print("95% confidence interval:", tuple(round(value, 2) for value in confidence_interval))

Mean liquidity: 27.11
95% confidence interval: (np.float64(25.35), np.float64(28.86))


## 5. Correlation

Correlation measures whether two numeric values move together. A negative liquidity-to-NPL correlation means liquidity tends to be lower when non-performing loans are higher.

In [9]:
pearson_corr, pearson_p = stats.pearsonr(clean_df["liquidity_ratio"], clean_df["npl_ratio"])
spearman_corr, spearman_p = stats.spearmanr(clean_df["liquidity_ratio"], clean_df["npl_ratio"])

print("Pearson correlation:", round(pearson_corr, 3), "p-value:", round(pearson_p, 4))
print("Spearman correlation:", round(spearman_corr, 3), "p-value:", round(spearman_p, 4))

Pearson correlation: -0.984 p-value: 0.0
Spearman correlation: -0.986 p-value: 0.0


## 6. Outliers with Z-Scores

A z-score tells us how far a value is from the average. A common beginner rule is to review values with an absolute z-score greater than 2.

In [10]:
clean_df["liquidity_z_score"] = stats.zscore(clean_df["liquidity_ratio"])
clean_df["liquidity_outlier_flag"] = clean_df["liquidity_z_score"].abs() > 2

outlier_columns = ["date", "institution", "liquidity_ratio", "liquidity_z_score", "liquidity_outlier_flag"]
display(clean_df.loc[clean_df["liquidity_outlier_flag"], outlier_columns])

,date,institution,liquidity_ratio,liquidity_z_score,liquidity_outlier_flag


## 7. Hypothesis Test

Question: do stressed and non-stressed observations have different average liquidity? We use an independent t-test because we compare two separate groups.

In [11]:
stressed = clean_df.loc[clean_df["stress_flag"] == 1, "liquidity_ratio"]
not_stressed = clean_df.loc[clean_df["stress_flag"] == 0, "liquidity_ratio"]

t_stat, p_value = stats.ttest_ind(stressed, not_stressed, equal_var=False)

print("Stressed mean liquidity:", round(stressed.mean(), 2))
print("Not stressed mean liquidity:", round(not_stressed.mean(), 2))
print("t-statistic:", round(t_stat, 3))
print("p-value:", round(p_value, 4))

if p_value < 0.05:
    print("Interpretation: the difference is statistically significant at the 5% level.")
else:
    print("Interpretation: the difference is not statistically significant at the 5% level.")

Stressed mean liquidity: 21.1
Not stressed mean liquidity: 30.35
t-statistic: -10.248
p-value: 0.0
Interpretation: the difference is statistically significant at the 5% level.


## 8. Export the Statistics

Export files make the notebook results usable in reports and later modules.

In [12]:
summary_path = OUTPUT_DIR / "beginner_scipy_summary.csv"
institution_summary_path = OUTPUT_DIR / "beginner_scipy_institution_summary.csv"

clean_df[numeric_cols].describe().T.round(2).to_csv(summary_path)
institution_summary.to_csv(institution_summary_path)

print("Saved:", summary_path)
print("Saved:", institution_summary_path)

Saved: outputs/beginner_scipy_summary.csv
Saved: outputs/beginner_scipy_institution_summary.csv
